#Notebook 1: Text Detection (YOLO)

Train YOLO to detect text regions in images.

In [ ]:
# Setup
import os, sys
sys.path.insert(0, '..')

from sklearn.model_selection import train_test_split
from ultralytics import YOLO
import yaml

from src.dataset import parse_xml, to_yolo_format, save_yolo_data

In [ ]:
# Download dataset (run once)
!gdown 15bTQg7W2NXg68ERJDSpY1t7EtL8eJ7az
!mkdir -p ../datasets && unzip -q icdar2003.zip -d ../datasets

In [ ]:
# Parse XML and convert to YOLO format
DATASET_DIR = "../datasets/SceneTrialTrain"

paths, sizes, labels, bboxes = parse_xml(f"{DATASET_DIR}/words.xml")
yolo_data = to_yolo_format(paths, sizes, bboxes)

print(f"Total images: {len(yolo_data)}")

In [ ]:
# Split and save
train_data, val_data = train_test_split(yolo_data, test_size=0.2, random_state=42)

YOLO_DIR = "../yolo_data"
save_yolo_data(train_data, "train", YOLO_DIR, DATASET_DIR)
save_yolo_data(val_data, "val", YOLO_DIR, DATASET_DIR)

# Create config
config = {
    "path": os.path.abspath(YOLO_DIR),
    "train": "train/images",
    "val": "val/images",
    "nc": 1,
    "names": ["text"]
}
with open(f"{YOLO_DIR}/data.yml", "w") as f:
    yaml.dump(config, f)

print(f"Train: {len(train_data)}, Val: {len(val_data)}")

In [ ]:
# Train YOLO
model = YOLO("yolo11m.pt")

model.train(
    data=f"{YOLO_DIR}/data.yml",
    epochs=80,
    imgsz=640,
    patience=20,
)

In [ ]:
# Evaluate
model = YOLO("../runs/detect/train/weights/best.pt")
model.val()

YOLO model saved to `runs/detect/train/weights/best.pt`

Next: Run `02_recognition.ipynb`